In [ ]:
# Random Forest — TOC prediction (calibration / training)
# Trains and evaluates the Random Forest model for TOC on the calibration dataset.
# The step applying the trained model to the full 318-well log dataset is not
# included here; those logs are publicly available from the WOGCC.

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import randint
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import RandomizedSearchCV, KFold
from sklearn.metrics import r2_score, mean_absolute_error, mean_absolute_percentage_error

# Load calibration data (place the file in the same folder, or update the path)
file_path = 'TOC ML modeling data.xlsx'
df = pd.read_excel(file_path, sheet_name='Sheet1')
df = df.dropna(subset=['ResDeep_N', 'RHOB_N', 'GR_N', 'TOC'])

X = df[['ResDeep_N', 'RHOB_N', 'GR_N']].values
y = df['TOC'].values
toc_mean = np.mean(y)

# Hyperparameter tuning and fit
rf = RandomForestRegressor(random_state=42)
param_dist = {
    'n_estimators': randint(100, 300),
    'max_depth': randint(3, 10),
    'max_features': ['sqrt', 'log2']
}
random_search = RandomizedSearchCV(
    rf, param_distributions=param_dist,
    n_iter=120, scoring='r2', cv=5,
    verbose=1, n_jobs=-1, random_state=42
)
random_search.fit(X, y)
best_rf = random_search.best_estimator_

# Calibration performance
y_pred = best_rf.predict(X)
r2 = r2_score(y, y_pred)
mae = mean_absolute_error(y, y_pred)
print("Best hyperparameters:", random_search.best_params_)
print(f"[Calibration] R2 = {r2:.4f} | MAE = {mae:.2f} wt.% | Mean TOC = {toc_mean:.4f} wt.%")

# Predicted vs observed (calibration)
plt.figure(figsize=(7, 6))
plt.scatter(y, y_pred, color='darkgreen', edgecolors='k', label='Samples')
plt.plot([min(y), max(y)], [min(y), max(y)], 'r--', label='45 deg reference')
plt.xlabel('Observed TOC (wt. %)')
plt.ylabel('Predicted TOC (wt. %)')
plt.title('Predicted vs Observed TOC (Random Forest)')
plt.legend(); plt.grid(True)
plt.text(0.025, 0.85,
         f"R2 = {r2:.4f}\nMAE = {mae:.2f} wt.%\nMean TOC = {toc_mean:.4f} wt.%",
         transform=plt.gca().transAxes, fontsize=12, verticalalignment='top',
         bbox=dict(facecolor='white', alpha=0.7))
plt.tight_layout()
plt.show()

# 5-fold cross-validation
kf = KFold(n_splits=5, shuffle=True, random_state=42)
r2_scores, mae_scores = [], []

plt.figure(figsize=(15, 12))
for fold, (train_idx, test_idx) in enumerate(kf.split(X)):
    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]

    best_rf.fit(X_train, y_train)
    y_pred_fold = best_rf.predict(X_test)

    r2_fold = r2_score(y_test, y_pred_fold)
    mae_fold = mean_absolute_error(y_test, y_pred_fold)
    r2_scores.append(r2_fold)
    mae_scores.append(mae_fold)


    plt.subplot(3, 2, fold + 1)
    plt.scatter(y_test, y_pred_fold, edgecolors='k')
    plt.plot([min(y), max(y)], [min(y), max(y)], 'r--', label='45 deg reference')
    plt.xlabel('Observed TOC (wt. %)')
    plt.ylabel('Predicted TOC (wt. %)')
    plt.title(f'Fold {fold + 1}: Predicted vs Observed')
    plt.grid(True); plt.legend()
    plt.text(0.02, 0.95,
             f"Test MAE = {mae_fold:.2f} wt.%\nTest Mean = {np.mean(y_test):.2f} wt.%",
             transform=plt.gca().transAxes, fontsize=12, verticalalignment='top',
             bbox=dict(facecolor='white', alpha=0.7))

plt.tight_layout()
plt.suptitle('Cross-Validation: Predicted vs Observed TOC', fontsize=16, y=1.02)
plt.show()

print("\nCross-Validation Summary (test sets):")
print(f"MAE Mean +/- Std = {np.mean(mae_scores):.2f} +/- {np.std(mae_scores):.2f}")